# Cell 1: load the inventory and select series

In [ ]:
import os, glob
import numpy as np
import pandas as pd
import pydicom
from pathlib import Path

ROOT = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")

def find_file(name):
    for top in os.listdir("/kaggle/input"):
        if top == "competitions":
            continue
        hits = glob.glob(f"/kaggle/input/{top}/**/{name}", recursive=True)
        if hits:
            return hits[0]
    raise FileNotFoundError(name)

inv = pd.read_csv(find_file("series_inventory.csv"))
study = pd.read_csv(find_file("study_inventory.csv"))
print(find_file("series_inventory.csv"), inv.shape, study.shape)

sel = (inv.assign(thick_gap=(inv["thick_mm"] - 3).abs())
          .sort_values(["Fluid_Sensitive", "thick_gap", "rows"], ascending=[False, True, False])
          .drop_duplicates(["StudyInstanceUID", "Anatomical_Plane"])
          .sort_values(["StudyInstanceUID", "Anatomical_Plane"])
          .reset_index(drop=True))

print("\nselected:", len(sel), " expected:", 3 * len(study))
print(pd.crosstab(sel["Anatomical_Plane"], sel["Fluid_Sensitive"]))
print("\nselected n_files:", sel["n_files"].describe().round(1).to_dict())
print("selected with > 80 slices:", (sel["n_files"] > 80).sum())
print("GB to read:", round(sel["mb"].sum() / 1e3, 1))

# Cell 2: laterality and orientation on 10 pilot studies

In [ ]:
pilot_ids = study["StudyInstanceUID"].sample(10, random_state=1).tolist()
pilot = sel[sel["StudyInstanceUID"].isin(pilot_ids)]

TAGS = ["Laterality", "ImageLaterality", "BodyPartExamined", "StudyDescription", "SeriesDescription"]

def axis_name(v):
    i = int(np.argmax(np.abs(v)))
    return ["RL", "AP", "IS"][i][int(v[i] > 0)]

rows = []
for r in pilot.itertuples(index=False):
    d = ROOT / "train_series" / r.StudyInstanceUID / r.SeriesInstanceUID
    ds = pydicom.dcmread(next(os.scandir(d)).path, stop_before_pixels=True)
    iop = np.array(ds.ImageOrientationPatient, dtype=float)
    out = {"study": r.StudyInstanceUID[-6:], "plane": r.Anatomical_Plane[:3],
           "fs": r.Fluid_Sensitive, "n": r.n_files,
           "row_dir": axis_name(iop[:3]), "col_dir": axis_name(iop[3:])}
    for t in TAGS:
        out[t] = ds.get(t)
    rows.append(out)
print(pd.DataFrame(rows).to_string())

# Cell 3: orientation and laterality tags for all selected series

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def header_info(r):
    d = ROOT / "train_series" / r.StudyInstanceUID / r.SeriesInstanceUID
    ds = pydicom.dcmread(next(os.scandir(d)).path, stop_before_pixels=True)
    iop = np.array(ds.ImageOrientationPatient, dtype=float)
    lat = ds.get("Laterality") or ds.get("ImageLaterality") or ""
    return {"SeriesInstanceUID": r.SeriesInstanceUID, "lat_tag": str(lat).strip().upper(),
            "row_dir": axis_name(iop[:3]), "col_dir": axis_name(iop[3:])}

with ThreadPoolExecutor(16) as ex:
    hdr = pd.DataFrame(list(ex.map(header_info, sel.itertuples(index=False))))
sel = sel.drop(columns=hdr.columns[1:], errors="ignore").merge(hdr, on="SeriesInstanceUID")

print(sel.groupby(["Anatomical_Plane", "row_dir", "col_dir"]).size().to_string())
print("\n" + sel["lat_tag"].replace("", "(blank)").value_counts().to_string())

study_lat = sel.groupby("StudyInstanceUID")["lat_tag"].agg(
    lambda s: "/".join(sorted(set(s) - {""})) or "none")
print("\nper study:\n" + study_lat.value_counts().to_string())

# Cell 4: side mentioned in the report

In [ ]:
import re

reports = (pd.read_csv(ROOT / "train.csv", usecols=["StudyInstanceUID", "Report"])
             .set_index("StudyInstanceUID")["Report"].str.lower())

SIDE = {
    "L": r"\b(left|izquierd[ao]|links|linke\w*|gauche|lijev\w*)\b",
    "R": r"\b(right|derech[ao]|rechts|rechte\w*|droite?|desn\w*)\b",
}

def side_of(text):
    l = re.search(SIDE["L"], text) is not None
    r = re.search(SIDE["R"], text) is not None
    return "both" if l and r else "L" if l else "R" if r else "none"

rep_side = reports.map(side_of)
lang = study.set_index("StudyInstanceUID")["lang"]
print(pd.crosstab(lang, rep_side).to_string())
print("\n" + pd.crosstab(study_lat, rep_side.reindex(study_lat.index)).to_string())

# Cell 5: loader and visual check on the 10 pilot studies

In [ ]:
import time
import matplotlib.pyplot as plt

def load_volume(study_uid, series_uid, split="train_series"):
    d = ROOT / split / study_uid / series_uid
    dss = [pydicom.dcmread(e.path) for e in os.scandir(d)]
    qa = {"n": len(dss), "sort": "position"}
    iop = np.array(dss[0].ImageOrientationPatient, dtype=float)
    normal = np.cross(iop[:3], iop[3:])
    if normal[np.argmax(np.abs(normal))] < 0:
        normal = -normal
    try:
        pos = np.array([normal @ np.array(ds.ImagePositionPatient, dtype=float) for ds in dss])
    except AttributeError:
        pos = np.array([float(ds.InstanceNumber) for ds in dss])
        qa["sort"] = "instance"
    order = np.argsort(pos, kind="stable")
    gaps = np.diff(pos[order])
    med = float(np.median(gaps)) if len(gaps) else 0.0
    qa["spacing"] = round(med, 3)
    qa["dup_pos"] = int((gaps < 1e-3).sum())
    qa["big_gaps"] = int((gaps > 1.5 * med).sum()) if med > 0 else 0
    arrs = [dss[i].pixel_array.astype(np.float32) for i in order]
    qa["mixed_shapes"] = len({a.shape for a in arrs}) > 1
    return (None if qa["mixed_shapes"] else np.stack(arrs)), qa

def window(img):
    lo, hi = np.percentile(img, [0.5, 99.5])
    return np.clip((img - lo) / max(hi - lo, 1e-6), 0, 1)

panels = [("Sagittal", 0.5), ("Coronal", 0.4), ("Coronal", 0.6), ("Coronal", 0.8), ("Axial", 0.5)]
fig, axes = plt.subplots(len(pilot_ids), 5, figsize=(15, 3 * len(pilot_ids)))
qa_rows, t0 = [], time.time()
for i, sid in enumerate(pilot_ids):
    vols = {}
    for r in sel[sel["StudyInstanceUID"] == sid].itertuples(index=False):
        vols[r.Anatomical_Plane], qa = load_volume(sid, r.SeriesInstanceUID)
        qa_rows.append({"study": sid[-6:], "plane": r.Anatomical_Plane[:3], **qa})
    for j, (p, f) in enumerate(panels):
        ax, v = axes[i, j], vols.get(p)
        if v is not None:
            ax.imshow(window(v[int(f * (len(v) - 1))]), cmap="gray")
        ax.set_title(f"{sid[-6:]} {p[:3]} {f} tag={study_lat[sid]} rep={rep_side[sid]}", fontsize=8)
        ax.axis("off")
print(f"loaded {len(pilot_ids)} studies in {time.time() - t0:.1f}s (single thread)")
print(pd.DataFrame(qa_rows).to_string())
plt.tight_layout()
plt.show()

# Cell 6: does geometry give us laterality?

In [ ]:
def coronal_center_x(r):
    d = ROOT / "train_series" / r.StudyInstanceUID / r.SeriesInstanceUID
    ds = pydicom.dcmread(next(os.scandir(d)).path, stop_before_pixels=True)
    iop = np.array(ds.ImageOrientationPatient, dtype=float)
    ipp = np.array(ds.ImagePositionPatient, dtype=float)
    px = np.array(ds.PixelSpacing, dtype=float)
    c = ipp + iop[:3] * ds.Columns / 2 * px[1] + iop[3:] * ds.Rows / 2 * px[0]
    return {"StudyInstanceUID": r.StudyInstanceUID, "center_x": c[0]}

cor = sel[sel["Anatomical_Plane"] == "Coronal"]
with ThreadPoolExecutor(16) as ex:
    cx = pd.DataFrame(list(ex.map(coronal_center_x, cor.itertuples(index=False))))
cx = cx.set_index("StudyInstanceUID")["center_x"]

side = study_lat.replace({"LEFT": "L", "RIGHT": "R"})
print(cx.groupby(side).describe().round(1).to_string())

k = side[side.isin(["L", "R"])].index
agree = (cx[k] > 0) == (side[k] == "L")
vend = study.set_index("StudyInstanceUID")["vendor"]
print("\noverall agreement:", round(agree.mean(), 3))
print(agree.groupby(vend[k]).agg(["mean", "size"]).round(3).to_string())

# Cell 7: preprocess 10 fresh studies, measure time and size, visual check

In [ ]:
import torch
import torch.nn.functional as F

D, S = 32, 256

def normalize(vol):
    lo, hi = np.percentile(vol, [0.5, 99.5])
    return np.clip((vol - lo) / max(hi - lo, 1e-6), 0, 1).astype(np.float32)

def pad_square(vol):
    d, h, w = vol.shape
    m = max(h, w)
    out = np.zeros((d, m, m), dtype=vol.dtype)
    y, x = (m - h) // 2, (m - w) // 2
    out[:, y:y + h, x:x + w] = vol
    return out

def resample(vol):
    t = torch.from_numpy(pad_square(vol))[:, None]
    t = F.interpolate(t, size=(S, S), mode="bilinear", antialias=True, align_corners=False)[:, 0]
    t = t.reshape(t.shape[0], S * S).T[None]
    t = F.interpolate(t, size=D, mode="linear", align_corners=False)
    return (t[0].T.reshape(D, S, S).clamp(0, 1) * 255).round().to(torch.uint8).numpy()

def process_study(rows, split="train_series"):
    vols, qas = {}, []
    for r in rows.itertuples(index=False):
        vol, qa = load_volume(r.StudyInstanceUID, r.SeriesInstanceUID, split)
        key = r.Anatomical_Plane[:3].lower()
        qa.update(study=r.StudyInstanceUID[-6:], plane=key)
        if vol is not None:
            vols[key] = resample(normalize(vol))
        qas.append(qa)
    return vols, qas

bench_ids = study["StudyInstanceUID"].sample(10, random_state=2).tolist()
out_dir = Path("/kaggle/working/pilot_cache")
out_dir.mkdir(exist_ok=True)

t0, qa_all = time.time(), []
for sid in bench_ids:
    vols, qas = process_study(sel[sel["StudyInstanceUID"] == sid])
    np.savez_compressed(out_dir / f"{sid}.npz", **vols)
    qa_all += qas
el = time.time() - t0

mb = sum(f.stat().st_size for f in out_dir.glob("*.npz")) / 1e6
raw = len(bench_ids) * 3 * D * S * S / 1e6
print(f"{el / len(bench_ids):.2f} s/study single thread")
print(f"compressed {mb:.1f} MB vs raw {raw:.1f} MB -> full train about {mb / len(bench_ids) * 4407 / 1e3:.1f} GB")
print(pd.DataFrame(qa_all).to_string())

fig, axes = plt.subplots(len(bench_ids), 3, figsize=(9, 3 * len(bench_ids)))
for i, sid in enumerate(bench_ids):
    z = np.load(out_dir / f"{sid}.npz")
    for j, key in enumerate(["sag", "cor", "axi"]):
        axes[i, j].imshow(z[key][D // 2], cmap="gray", vmin=0, vmax=255)
        axes[i, j].set_title(f"{sid[-6:]} {key} {z[key].shape}", fontsize=8)
        axes[i, j].axis("off")
plt.tight_layout()
plt.show()

# Cell 8: resolve laterality and save it

In [ ]:
side = study_lat.replace({"LEFT": "L", "RIGHT": "R"})
side = side.where(side.isin(["L", "R"]))
geo_ok = vend.reindex(cx.index).isin(["siemens", "philips"]) & (cx.abs() > 30)
geo = pd.Series(np.where(cx > 0, "L", "R"), index=cx.index).where(geo_ok)

lat = pd.DataFrame({"tag": side, "geo": geo, "center_x": cx, "vendor": vend, "report": rep_side})
lat["side"] = lat["tag"].fillna(lat["geo"])
lat["side_src"] = np.where(lat["tag"].notna(), "tag", np.where(lat["geo"].notna(), "geo", "unknown"))

print(lat["side_src"].value_counts().to_string())
print("\n" + pd.crosstab(lat["vendor"], lat["side_src"]).to_string())

g = lat[(lat["side_src"] == "geo") & lat["report"].isin(["L", "R"])]
print("\ngeo vs report on untagged studies:", round((g["side"] == g["report"]).mean(), 3), " n =", len(g))

lat.drop(columns=["report"]).rename_axis("StudyInstanceUID").reset_index().to_csv(
    "/kaggle/working/laterality.csv", index=False)

# Cell 9: 100 studies in parallel

In [ ]:
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor
from functools import partial

def init_worker():
    torch.set_num_threads(1)

def run_one(sid, out_dir, split="train_series"):
    try:
        vols, qas = process_study(sel[sel["StudyInstanceUID"] == sid], split)
        np.savez_compressed(out_dir / f"{sid}.npz", **vols)
        return qas, None
    except Exception as err:
        return [], (sid, repr(err)[:200])

pilot_dir = Path("/kaggle/working/pilot100")
pilot_dir.mkdir(exist_ok=True)
ids100 = study["StudyInstanceUID"].sample(100, random_state=3).tolist()

t0 = time.time()
with ProcessPoolExecutor(4, mp_context=mp.get_context("fork"), initializer=init_worker) as ex:
    results = list(ex.map(partial(run_one, out_dir=pilot_dir), ids100, chunksize=2))
el = time.time() - t0

qa100 = pd.DataFrame([q for qs, _ in results for q in qs])
errs = [e for _, e in results if e]
sizes = pd.Series({f.stem: f.stat().st_size / 1e6 for f in pilot_dir.glob("*.npz")})

print(f"{el / 100:.2f} s/study with 4 workers -> full train about {el / 100 * 4407 / 3600:.2f} h")
print(f"{sizes.mean():.2f} MB/study (max {sizes.max():.2f}) -> full train about {sizes.sum() / 100 * 4407 / 1e3:.1f} GB")
print("errors:", errs)
print(qa100[["dup_pos", "big_gaps", "mixed_shapes"]].astype(int).sum().to_string())
print("sorted by instance number:", (qa100["sort"] != "position").sum())
print("\nMB/study by vendor:\n" + sizes.groupby(vend.reindex(sizes.index)).mean().round(2).to_string())